# Z4 115M — Kaggle TPU smoke

This runs the measured **`kaggle10m_nomtp`** profile on a Kaggle TPU:

- 115,238,237 parameters (MTP compute skipped)
- bf16 autocast on XLA
- 600-second training cap
- WikiText-2 train+valid, 24k held-out test tokens

`TPU_MODE = "single"` uses one XLA device with `train_v3.py --device xla`.
`TPU_MODE = "multi"` uses the experimental 8-replica wrapper
`z3/train_tpu.py`.  Start with single; the multi-core wrapper is not yet
TPU-validated.

Training runs without gradient checkpointing on TPU: recomputation makes the
XLA graph much larger and slower to compile, and the 115M activations fit.

This is an early-training smoke, not a frontier claim.


In [ ]:
!nvidia-smi 2>/dev/null || true
import os, sys, subprocess, time, importlib.util
os.environ.setdefault("PJRT_DEVICE", "TPU")
print({k: os.environ.get(k) for k in ("TPU_NAME", "COLAB_TPU_ADDR", "PJRT_DEVICE")}, flush=True)
print("image", os.environ.get("COLAB_IMAGE_TYPE"), "vfio", os.path.exists("/dev/vfio"), flush=True)
subprocess.run([sys.executable, "-c", "import torch; print('image torch', torch.__version__)"], timeout=120)

# A real TPU VM ships /dev/vfio and a matching torch_xla.  Use it as-is;
# only install a pinned CPU torch stack when torch_xla is missing.  Kaggle's
# non-TPU image reports torch 2.11, while the libtpu index stops at 2.9.
if importlib.util.find_spec("torch_xla") is None or not os.path.exists("/dev/vfio"):
    steps = [
        [sys.executable, "-m", "pip", "install", "-q", "torch==2.9.0",
         "--index-url", "https://download.pytorch.org/whl/cpu"],
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "torch_xla==2.9.0",
         "-f", "https://storage.googleapis.com/libtpu-releases/index.html"],
        [sys.executable, "-m", "pip", "install", "-q", "libtpu==0.0.21.1",
         "-f", "https://storage.googleapis.com/libtpu-releases/index.html"],
        [sys.executable, "-m", "pip", "install", "-q", "absl-py", "numpy", "pyyaml", "requests", "tpu-info"],
    ]
else:
    steps = []
    print("using the preinstalled TPU torch_xla stack", flush=True)
for command in steps:
    print(f"$ {' '.join(command)}", flush=True)
    t0 = time.time()
    subprocess.run(command, check=True, timeout=900)
    print(f"  done in {time.time()-t0:.0f}s", flush=True)

checks = [
    ("torch", "import torch; print('torch', torch.__version__)", 120),
    ("torch_xla", "import torch_xla; print('torch_xla', torch_xla.__version__)", 300),
    ("devices", "import torch_xla.core.xla_model as xm; print('devices', xm.get_xla_supported_devices())", 300),
    ("device", "import torch, torch_xla.core.xla_model as xm; d=xm.xla_device(); print('device', d); x=torch.ones(8,8,device=d); print('sum', float((x+x).sum().cpu()))", 900),
]
for name, snippet, limit in checks:
    print(f"[check] {name} (limit {limit}s)", flush=True)
    t0 = time.time()
    subprocess.run([sys.executable, "-c", snippet], check=True, timeout=limit)
    print(f"[check] {name} ok in {time.time()-t0:.0f}s", flush=True)


In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO = Path("/kaggle/working/nanlm_experimental")
if not (REPO / "z3" / "train_v3.py").is_file():
    subprocess.run(["git","clone","--depth","1","https://github.com/shemshallah/nanlm_experimental.git",str(REPO)],check=True)
else:
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
os.chdir(REPO)
print(REPO)
subprocess.run(["git","-C",str(REPO),"rev-parse","HEAD"])
subprocess.run(["git","-C",str(REPO),"log","-1","--oneline"])


In [ ]:
subprocess.run([sys.executable,"-m","pip","install","-q","tokenizers","numpy","pyarrow"],check=True)


In [ ]:
import urllib.request
Path("data").mkdir(exist_ok=True)
parts=[]
for split in ("train","valid"):
    url=f"https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/{split}.txt"
    dest=Path(f"data/wikitext_{split}.txt")
    urllib.request.urlretrieve(url,dest)
    parts.append(dest.read_text(encoding="utf-8"))
corpus=Path("data/wikitext_corpus.txt")
corpus.write_text("\n\n".join(parts),encoding="utf-8")
print(corpus, corpus.stat().st_size)
subprocess.run([
    sys.executable,"z3/build_domain_bins.py",
    "--corpus","wikitext=data/wikitext_corpus.txt",
    "--out","data/wikitext_bins",
    "--max-bytes-per-domain","12000000",
    "--train-fraction","0.98","--valid-fraction","0.01",
],check=True)


In [ ]:
TPU_MODE = "single"   # "single" or "multi"
OUT = Path("/kaggle/working/runs/z3_115m_tpu")
sources = [
    "--preset","kaggle10m_nomtp",
    "--corpus","data/wikitext_bins/train.bin",
    "--val-corpus","data/wikitext_bins/valid.bin",
    "--data-manifest","data/wikitext_bins/manifest.json",
    "--batch-size","4",
    "--no-grad-checkpoint",
]
if TPU_MODE == "single":
    cmd=[sys.executable,"z3/train_v3.py","--device","xla","--amp",*sources,"--out",str(OUT)]
else:
    cmd=[sys.executable,"z3/train_tpu.py","--nprocs","8","--out",str(OUT),*sources]
print(" ".join(cmd), flush=True)
# Hard host-side cap: a stuck XLA compile must surface as an error, not burn
# the whole TPU session.  The in-process --max-seconds cap covers training.
try:
    subprocess.run(cmd, check=True, timeout=1800)
except subprocess.TimeoutExpired:
    raise RuntimeError("training subprocess exceeded 1800s; see log above")
except subprocess.CalledProcessError:
    print("TPU launch failed; retrying the single-device path", flush=True)
    single=[sys.executable,"z3/train_v3.py","--device","xla","--amp",*sources,"--out",str(OUT)]
    subprocess.run(single, check=True, timeout=1800)


In [ ]:
CKPT = OUT/"best.pt"
if not CKPT.is_file():
    CKPT = OUT/"final.pt"
print("evaluating", CKPT, flush=True)
subprocess.run([
    sys.executable,"z3/eval_domains.py",
    "--checkpoint",str(CKPT),
    "--test-dir","data/wikitext_bins/test",
    "--data-manifest","data/wikitext_bins/manifest.json",
    "--tokenizer","layered","--document-framing","--sidecar","--require-sidecar",
    "--ctx","512","--batch-size","4","--device","xla",
    "--out",str(OUT/"test_domains.json"),
],check=True,timeout=900)


In [ ]:
import json
result=json.loads((OUT/"result.json").read_text())
test=json.loads((OUT/"test_domains.json").read_text())
print(json.dumps({
    "params":result.get("params"),
    "steps":result.get("steps_completed"),
    "tokens_seen":result.get("tokens_seen"),
    "nonfinite_skips":result.get("nonfinite_skips"),
    "best_step":(result.get("best") or {}).get("step"),
    "best_bpb":((result.get("best") or {}).get("metrics") or {}).get("bits_per_byte"),
    "final_bpb":(result.get("final") or {}).get("bits_per_byte"),
    "macro_test_bpb":test.get("macro_bits_per_byte"),
    "feedback":(result.get("final") or {}).get("feedback"),
},indent=2,sort_keys=True))
